In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold, train_test_split

train = pd.read_csv('../data/raw/train.csv')
print(f"Shape: {train.shape}")

In [ ]:
# Drop outliers identified in EDA
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]

# Log-transform target
y = np.log1p(train['SalePrice'])
X = train.drop(['SalePrice', 'Id'], axis=1)

In [ ]:
print(f"Shape after dropping outliers: {train.shape}")
print(f"X: {X.shape}, y: {y.shape}")

In [ ]:
# 1. Fill numeric NaN with median
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())

# 2. Fill categorical NaN with 'Missing' (to remove NaN)
cat_cols = X.select_dtypes(include=['object']).columns
X[cat_cols] = X[cat_cols].fillna('Missing')

# 3. One-hot encode categoricals
X = pd.get_dummies(X, drop_first=True)

In [ ]:
print(f"After processing: {X.shape}")
print(f"NaN count: {X.isnull().sum().sum()}")

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

dt = DecisionTreeRegressor(random_state=42)
dt_scores = -cross_val_score(dt, X, y, cv=kf, scoring='neg_root_mean_squared_error')

print(f"Decision Tree RMSE: {dt_scores.mean():.4f} \nStd: {dt_scores.std():.4f}")

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_scores = -cross_val_score(rf, X, y, cv=kf, scoring='neg_root_mean_squared_error')

print(f"Random Forest RMSE: {rf_scores.mean():.4f} \nStd: {rf_scores.std():.4f}")

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

ridge = Ridge(alpha=0.1)
ridge_scores = -cross_val_score(ridge, X_scaled, y, cv=kf, scoring='neg_root_mean_squared_error')

print(f"Ridge RMSE: {ridge_scores.mean():.4f} \nStd: {ridge_scores.std():.4f}")

## Summary

We tested 3 models (Decision Tree, Random Forest, Ridge - linear model)


Decision Tree: RMSE: 0.2073, Std: 0.0096 
Random Forest: RMSE: 0.1397, Std: 0.0101
Ridge:         RMSE: 0.1299, Std: 0.0133

Ridge has the lowest RMSE scores and wins at this stage
Tree-based models may catch up after feature engineering and tuning